# Example Usage - Differential Evolution (DE)

In [ ]:
%load_ext autoreload
%autoreload 2

Set up logger

In [ ]:
import logging

logging.basicConfig(
    encoding="utf-8",
    filemode="a",
    format="[{asctime}] [{levelname}] {message}",
    style="{",
    datefmt="%Y-%m-%d, %H:%M",
    level=logging.INFO,
    force=True,
)

logger = logging.getLogger()
logger.info("Hello logging!")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

Lets First define a problem:

In [ ]:
rng = np.random.default_rng(0)

x = np.linspace(0, 10, 500)
y = np.cos(x) + rng.normal(0, 0.2, 500)

In [ ]:
def rmse(y, y_pred):
    return np.sqrt(sum((y - y_pred)**2) / len(y))

Next, lets make a population which we will evolve:

In [ ]:
from pyeas.population import Genes, Population

member = [
    Genes(bounds=(-5,5), number=6),
]

pop = Population(
    size=40,
    member=member,
    seed=42,
)

pop

In [ ]:
pop.population[0][0]

In [ ]:
pop._normalised_population[0][0]

In [ ]:
pop.denormalise(pop._normalised_population)

### The First Generation

Now, lets initialize our solver, and iterate though the fist generation of population members:

In [ ]:
from pyeas.algorithms import DE 

optimizer1 = DE(
    population=pop,
    mut=0.6,
    crossp=0.6,
    mut_scheme = 'ttb1',  # 'ttb1', rand1
    seed=1,
)

In [ ]:
trial_pop = optimizer1.ask(loop=0)
print(np.shape(trial_pop))

trial_pop[:3]

In [ ]:
from pyeas.examples.funcs import polynomial_order_5

solutions = []
for t, _trial in enumerate(trial_pop):
    _pred = polynomial_order_5(x, _trial)
    _value = rmse(y, _pred)
    solutions.append(_value)
solutions[:5]  # print(t, value)

In [ ]:
optimizer1.tell(solutions, trial_pop)

In [ ]:
optimizer1.population.population[optimizer1._best_idx]

### Generational Optimization

Starting from scratch:

In [ ]:
optimizer = DE(
    population=pop, 
    mut=0.4, 
    crossp=0.4, 
    mut_scheme='best2', # 'ttb1', rand1
    seed=1
)  

In [ ]:
from tqdm import tqdm

num_gens = 1000
pbar = tqdm(range(num_gens), unit=' generations')

for generation in pbar:
    trial_pop_1 = optimizer.ask(loop=generation)

    solutions_1 = []
    for _trial in trial_pop_1:
        _pred = polynomial_order_5(x, _trial)
        _value = rmse(y, _pred)
        solutions_1.append(_value)

    optimizer.tell(solutions_1, trial_pop_1)
    
    pbar.set_description_str(f'Best Member: {optimizer.best_member.member}, loss: {optimizer.best_member.loss:.4} ')

Consider the population change over time:

In [ ]:
_solutions = list(zip(*optimizer.history.parents))
_trial_m = list(zip(*optimizer.history.get_trial_means()))

_fig, _axs = plt.subplots(
    nrows=len(_solutions), 
    sharex=True,
    figsize=(8,len(_solutions)*1.1),
)
for _i in range(len(_solutions)):
    _axs[_i].plot(_solutions[_i])
    _axs[_i].plot(_trial_m[_i], alpha=0.5)
    _axs[_i].set_ylabel(f'Gene {_i}')
    _axs[_i].set_ylim(pop.bounds[_i])

_axs[-1].set_xlabel('Generation')

Now, lets plot the performance over the generations:

In [ ]:
fig_performance, ax_performance = plt.subplots()
ax_performance.plot(optimizer.history.parent_losses)
ax_performance.set_yscale('log')
ax_performance.set_xlabel('Generation')
ax_performance.set_ylabel('loss')

# Create the twin axis
ax2 = ax_performance.twiny()

# Ensure the limits match if the points are meant to line up
ax2.set_xlim(ax_performance.get_xlim())

# Set the ticks and labels for the top axis
ax2.set_xticks(np.arange(optimizer.generation)[::200])
ax2.set_xticklabels(optimizer.history.n_evals[::200])
ax2.set_xlabel('Number of Evaluations')

And the final solution:

In [ ]:
fig_solution, ax_solution = plt.subplots()

ax_solution.scatter(x, y, marker='.', color='r', alpha=0.7, label='Target data')
plt.plot(x, np.cos(x), '--', label='ideal cos(x)', color='k', alpha=0.5)

data = polynomial_order_5(x, optimizer.history.best.member)
ax_solution.plot(x, data, label='OpenAI-ES Solution')

ax_solution.legend()

Now, lets animate the results:

In [ ]:
import matplotlib.animation as animation

plt.rcParams['animation.html'] = 'jshtml'

fig_ani, (ax_ani, ax_ani_2) = plt.subplots(ncols=2, figsize=(9, 4))

fig_ani.suptitle('DE fitting a 5th order polynomial to noisy cos() data')

ax_ani.set_ylim([-10, 10])
ax_ani.scatter(x, y, marker='.', color='r')
ax_ani.set_xlabel('x')
ax_ani.set_ylabel('y')

ax_ani_2.set_yscale('log')
ax_ani_2.plot(optimizer.history.parent_losses)
ax_ani_2.set_xlabel('Generation')
ax_ani_2.set_ylabel('rmse')

it_line, = ax_ani_2.plot(
    [0, 0], 
    [np.min(optimizer.history.parent_losses), np.max(optimizer.history.parent_losses)], 
    markersize=5, 
    color='k', 
    alpha=0.5,
)
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
lines = []

# --- MODIFIED: Use the original index (j) for history lookups ---
def ani(i):

    # Check if the current frame index i is valid for the history lists
    if i >= num_gens:
        return # Skip this frame if it's out of range

    lim = 10 - i / num_gens * (10 - 4)
    ax_ani.set_ylim([-lim, lim])
    data = polynomial_order_5(x, optimizer.history.parents[i])
    line, = ax_ani.plot(x, data, alpha=0.4)
    lines.append(line)
    if len(lines) > 10:
        lines[0].remove()
        lines.pop(0)
    it_line.set_xdata([i, i])

# --- MODIFIED: Only pass every 10th index to frames ---
step = int(np.around(num_gens/20, -1))
frames_to_animate = np.arange(0, num_gens, step)
length = 20 # You may need to adjust this depending on the desired animation duration
FPS = len(frames_to_animate) / length # Calculate FPS based on the new number of frames

ani = animation.FuncAnimation(
    fig_ani, 
    ani, 
    frames=frames_to_animate, # Use the sliced array of indices
    interval=200 # Interval between frames in ms (50ms corresponds to 20 FPS)
)

In [ ]:
from IPython.display import display, HTML
   

vid = ani.to_html5_video()
display(HTML(vid))

In [ ]:

# writer = animation.PillowWriter(
#     fps=15,
#     metadata=dict(artist='Me'),
#     bitrate=1800,
# )
# ani.save('scatter.gif', writer=writer)